# Lecture 4: Classes and NumPy

**Keeping data and calculations together, and working with numerical arrays**

**Learning goals**

After this lecture, you should be able to:

1. explain the difference between a class and an object,
2. store data as attributes and calculations as methods,
3. create and inspect one-dimensional NumPy arrays,
4. perform calculations on a whole array at once, and
5. use slicing and logical indexing.

Classes can feel abstract at first. We use one small class and keep the focus on the basic pattern.

**Contents**

1. [A first class](#first-class)
2. [A simple economic class](#economic-class)
3. [NumPy arrays](#numpy-arrays)
4. [Calculations with arrays](#array-calculations)
5. [Indexing and selecting observations](#indexing)
6. [A small economic example](#economic-example)
7. [Small exercises](#small-exercises)
8. [Summary](#summary)

<a id="first-class"></a>
## 1. A first class

A **class** defines a new type of object. It combines:

- data, stored as **attributes**, and
- functions, stored as **methods**.

`self` refers to the particular object being used.

In [1]:
class Human:

    def __init__(self, name, height, weight):
        self.name = name
        self.height = height
        self.weight = weight

    def bmi(self):
        return self.weight / (self.height/100)**2

Creating an object from a class is called creating an **instance**.

In [2]:
jeppe = Human('Jeppe', 182, 80)

print(jeppe.name)
print(jeppe.height)
print(f'{jeppe.bmi():.2f}')

Jeppe
182
24.15


When `jeppe.bmi()` is called, Python automatically uses `jeppe` as `self`. The method can therefore use the attributes `height` and `weight` without receiving them as separate arguments.

<a id="economic-class"></a>
## 2. A simple economic class

Consider a consumer with Cobb-Douglas demand:

$$
x_1 = \alpha \frac{I}{p_1}, \qquad
x_2 = (1-\alpha)\frac{I}{p_2}.
$$

The parameters become attributes, and the calculation becomes a method.

In [3]:
class Consumer:

    def __init__(self, income=1_000, p1=20, p2=40, alpha=0.5):
        self.income = income
        self.p1 = p1
        self.p2 = p2
        self.alpha = alpha

    def demand(self):
        x1 = self.alpha * self.income / self.p1
        x2 = (1 - self.alpha) * self.income / self.p2
        return x1, x2

In [4]:
consumer = Consumer()
x1, x2 = consumer.demand()

print('Demand for good 1:', x1)
print('Demand for good 2:', x2)

Demand for good 1: 25.0
Demand for good 2: 12.5


Different objects can have different parameter values.

In [5]:
low_income = Consumer(income=800)
high_income = Consumer(income=1_600)

print(low_income.demand())
print(high_income.demand())

(20.0, 10.0)
(40.0, 20.0)


This is the basic pattern used later in economic models:

1. parameters are attributes set in `__init__`,
2. calculations are methods, and
3. several parameterizations can be stored as separate objects.

<a id="numpy-arrays"></a>
## 3. NumPy arrays

NumPy is the main Python package for numerical arrays.

In [6]:
import numpy as np

A NumPy array is similar to a list, but it is designed for numerical calculations.

In [7]:
income = np.array([180.0, 240.0, 320.0, 150.0, 410.0])

print(income)
print(type(income))
print('shape:', income.shape)
print('size:', income.size)
print('data type:', income.dtype)

[180. 240. 320. 150. 410.]
<class 'numpy.ndarray'>
shape: (5,)
size: 5
data type: float64


Useful functions create arrays without first writing a list.

In [8]:
print(np.zeros(4))
print(np.ones(4))
print(np.linspace(0, 10, 6))

[0. 0. 0. 0.]
[1. 1. 1. 1.]
[ 0.  2.  4.  6.  8. 10.]


<a id="array-calculations"></a>
## 4. Calculations with arrays

Arithmetic is applied element by element. No loop is needed.

In [9]:
prices = np.array([20.0, 8.0, 45.0])
quantities = np.array([3.0, 5.0, 2.0])

expenditure = prices * quantities

print(expenditure)
print('Total expenditure:', np.sum(expenditure))

[60. 40. 90.]
Total expenditure: 190.0


Standard mathematical functions also work on the entire array.

In [10]:
x = np.array([1.0, 4.0, 9.0, 16.0])

print(np.sqrt(x))
print(np.mean(x))
print(np.max(x))
print(np.argmax(x))

[1. 2. 3. 4.]
7.5
16.0
3


NumPy arrays can be multiplied by a single number. NumPy applies the operation to every element.

In [11]:
wages = np.array([180.0, 220.0, 260.0])
wages_after_increase = 1.02 * wages

print(wages_after_increase)

[183.6 224.4 265.2]


<a id="indexing"></a>
## 5. Indexing and selecting observations

One-dimensional arrays use the same indexing and slicing syntax as lists.

In [12]:
income = np.array([180.0, 240.0, 320.0, 150.0, 410.0])

print(income[0])
print(income[-1])
print(income[1:4])

180.0
410.0
[240. 320. 150.]


A comparison creates an array of booleans. It can be used to select observations.

In [13]:
above_250 = income > 250

print(above_250)
print(income[above_250])

[False False  True False  True]
[320. 410.]


The mean of a boolean array is the share of observations for which the condition is true.

In [14]:
print('Number above 250:', np.sum(above_250))
print('Share above 250:', np.mean(above_250))

Number above 250: 2
Share above 250: 0.4


A slice of a NumPy array is a **view** of the original array. Changing the view also changes the original array.

In [15]:
baseline = np.array([100.0, 104.0, 109.0])
scenario = baseline[:]

scenario[0] = 200.0

print('baseline =', baseline)
print('scenario =', scenario)

baseline = [200. 104. 109.]
scenario = [200. 104. 109.]


Use `.copy()` when the new array should be independent.

In [16]:
baseline = np.array([100.0, 104.0, 109.0])
scenario = baseline.copy()

scenario[0] = 200.0

print('baseline =', baseline)
print('scenario =', scenario)

baseline = [100. 104. 109.]
scenario = [200. 104. 109.]


<a id="economic-example"></a>
## 6. A small economic example

NumPy can compute all growth rates from a time series in one expression.

In [17]:
gdp = np.array([100.0, 103.0, 107.0, 110.0])
growth = gdp[1:] / gdp[:-1] - 1

print(growth)

[0.03       0.03883495 0.02803738]


Read the expression from left to right:

- `gdp[1:]` contains all observations except the first,
- `gdp[:-1]` contains all observations except the last, and
- NumPy divides the matching elements.

<a id="small-exercises"></a>
## 7. Small exercises

### Exercise 1: A firm as a class

A firm produces output according to $y = A\ell^{\gamma}$ and earns profit

$$
\pi(\ell) = py - w\ell.
$$

Complete the class below so that `production(ell)` returns output and `profit(ell)` returns profit.

In [ ]:
class Firm:

    def __init__(self, A=2.0, gamma=0.5, wage=1.0, price=1.0):
        self.A = A
        self.gamma = gamma
        self.wage = wage
        self.price = price

    def production(self, ell):
        # Complete this line
        return None

    def profit(self, ell):
        # Complete this line
        return None

firm = Firm()
print(firm.production(4.0))
print(firm.profit(4.0))

**One possible answer**

In [19]:
class Firm:

    def __init__(self, A=2.0, gamma=0.5, wage=1.0, price=1.0):
        self.A = A
        self.gamma = gamma
        self.wage = wage
        self.price = price

    def production(self, ell):
        return self.A * ell**self.gamma

    def profit(self, ell):
        return self.price * self.production(ell) - self.wage * ell

firm = Firm()
print(firm.production(4.0))
print(firm.profit(4.0))

4.0
0.0


### Exercise 2: Income data

Use NumPy to compute the mean income, select incomes above the mean and compute their share.

In [ ]:
income = np.array([180.0, 240.0, 320.0, 150.0, 410.0, 260.0])

# Write your code here

**One possible answer**

In [21]:
mean_income = np.mean(income)
above_mean = income > mean_income

print('Mean income:', mean_income)
print('Incomes above the mean:', income[above_mean])
print('Share above the mean:', np.mean(above_mean))

Mean income: 260.0
Incomes above the mean: [320. 410.]
Share above the mean: 0.3333333333333333


### Exercise 3: Growth rates

Use array slicing to compute all growth rates in the illustrative series below without a loop.

In [ ]:
index = np.array([100.0, 102.0, 105.0, 111.0])

# Write your code here

**One possible answer**

In [23]:
growth = index[1:] / index[:-1] - 1
print(growth)

[0.02       0.02941176 0.05714286]


<a id="summary"></a>
## 8. Summary

You have used:

1. classes, objects, attributes, methods and `self`,
2. a small class for an economic calculation,
3. one-dimensional NumPy arrays,
4. element-by-element calculations,
5. indexing, slicing and logical indexing,
6. views and copies, and
7. vectorized growth-rate calculations.

More advanced class features and higher-dimensional NumPy operations are deliberately left for later use, when a model or data task requires them.